# Week 4 Session 1: The pandas analytical workflow (El Nino mission)
**Forge Academy | Data Science Foundations**
*Week 4 Session 1 — Working with Pandas for analytical workflows*

Today we practise the full analytical workflow on a real question the world is asking right now.

**Mission question**

> During El Nino years, does rainy-season rainfall in Nigeria look different — especially North (Kano) versus South (Lagos) — and what should we calmly expect under today's strong 2026-27 El Nino?

Every step below serves that question: load, inspect, clean, transform, merge, group, plot, then conclude with honest caveats.

**By the end of this session you can:**
1. Follow a pandas analytical workflow from question to conclusion.
2. Load and inspect more than one CSV.
3. Filter rows with boolean masks and create new columns with `.loc`.
4. Merge tables on shared keys and check the grain (what one row means).
5. Use `groupby` to compare groups, draw 1 to 2 simple charts, and state findings carefully.

**Setup:** keep these files in the **same folder** as this notebook:
- `oni.csv`
- `nigeria_city_rainfall_monthly.csv`
- `nigeria_north_staple_prices_monthly.csv` (stretch challenge only)

Open the notebook in VS Code, select your Python kernel, and run cells from top to bottom. Cells marked **Your turn** are for you to write code.

**Session map**
| Part | Topic |
|---|---|
| 0 | Mission, El Nino context, and the workflow |
| 1 | Load the data |
| 2 | Inspect |
| 3 | Clean habits (missing values and duplicates) |
| 4 | Filter and transform (ENSO labels) |
| 5 | Merge ONI with rainfall |
| 6 | Groupby: answer the mission |
| 7 | Two light charts |
| 8 | Challenge, recap, homework, EDA project kickoff |

## 0. Mission, context, and the workflow

### Why this mission
As of late 2026, NOAA's Climate Prediction Center has an **El Nino Advisory** in place. The October 2026 discussion said El Nino continues to strengthen, with a strong-to-very strong event likely through January-March 2027.

- NOAA CPC ENSO Diagnostic Discussion: https://www.cpc.ncep.noaa.gov/products/analysis_monitoring/enso_advisory/ensodisc.shtml
- FEWS NET notes that El Nino impacts in West Africa often show up as dry spells in parts of the Sahel and north-east Nigeria: https://fews.net/global/special-report/october-2026

We will not claim that El Nino "causes" every wet or dry month in Nigeria. Weather has many drivers. Our job as analysts is to **measure what the historical record shows**, then speak carefully.

### What is the Oceanic Nino Index (ONI)?
ONI is a 3-month running average of sea-surface temperature anomalies in the central/eastern tropical Pacific (Nino 3.4 region).

| ONI value | Rough label |
|---|---|
| `oni >= 0.5` | El Nino |
| `oni <= -0.5` | La Nina |
| otherwise | Neutral |

A full episode usually needs several overlapping seasons above or below the threshold. Today we use the simple seasonal rule so the workflow stays clear.

### Our datasets
| File | What it holds |
|---|---|
| `oni.csv` | Seasonal ONI from NOAA CPC (year, month of season middle, season code, SST, ONI) |
| `nigeria_city_rainfall_monthly.csv` | Monthly rainfall (mm) for Lagos and Kano, 1950-2025 |
| `nigeria_north_staple_prices_monthly.csv` | Stretch only: northern Nigeria staple prices (per kg), from WFP via HDX |

**Attribution**
- ONI: NOAA Climate Prediction Center (public U.S. government data). Source text: https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt
- Rainfall: Weather data by Open-Meteo.com (CC BY 4.0). https://open-meteo.com/
- Staple prices (stretch): WFP via HDX, CC BY-IGO. https://data.humdata.org/dataset/wfp-food-prices-for-nigeria

See `README_data.txt` in this folder for more detail.

### The analytical workflow (today's spine)
1. **Question** — what are we trying to learn?
2. **Load** — bring tables into pandas.
3. **Inspect** — shape, types, missing values, strange values.
4. **Clean** — fix only what needs fixing, and say why.
5. **Transform** — add useful columns (for example an El Nino flag).
6. **Merge** — join tables on shared keys; check grain.
7. **Group / aggregate** — compare groups that answer the question.
8. **Plot** — a few charts that serve the question (not chart-for-chart's-sake).
9. **Conclude** — findings plus caveats (correlation is not causation).

## 1. Load the data

Import pandas (and matplotlib for two light charts later).

In [ ]:
import pandas as pd

In [ ]:
import matplotlib.pyplot as plt

Load the Oceanic Nino Index.

In [ ]:
oni = pd.read_csv("oni.csv")

In [ ]:
oni.head()

Load the Lagos and Kano monthly rainfall.

In [ ]:
rain = pd.read_csv("nigeria_city_rainfall_monthly.csv")

In [ ]:
rain.head()

If you get `FileNotFoundError`, the CSV is not in the same folder as this notebook. Move the three CSV files next to this notebook and run the load cells again.

`head()` shows the first 5 rows. `tail()` shows the last rows.

In [ ]:
oni.tail()

In [ ]:
rain.tail()

### Your turn 1
Load only the first 6 rows of `oni.csv` into a variable called `oni_peek`, using `nrows=6`. Then display it.

In [ ]:
# Your code here

## 2. Inspect

Good analysis starts with looking. We check shape, columns, types, missing values, and a few summaries before we change anything.

In [ ]:
print("ONI rows and columns:", oni.shape)

In [ ]:
print("Rain rows and columns:", rain.shape)

In [ ]:
print(list(oni.columns))

In [ ]:
print(list(rain.columns))

`info()` shows types and non-null counts. Scan for surprises.

In [ ]:
oni.info()

In [ ]:
rain.info()

**What does one row mean (grain)?**
- In `oni`, one row is one overlapping 3-month season. `month` is the middle month of that season (for example JJA -> July).
- In `rain`, one row is one city in one calendar month.

We will join them on `year` and `month`. That only works because we gave ONI a middle-month key on purpose.

In [ ]:
oni.describe()

In [ ]:
rain.describe()

How many rows per city? `value_counts()` answers that quickly.

In [ ]:
rain["city"].value_counts()

What years does each file cover?

In [ ]:
print(oni["year"].min(), "to", oni["year"].max())

In [ ]:
print(rain["year"].min(), "to", rain["year"].max())

Look at the most recent ONI values. The 2026 seasons tell us where we are in the current event.

In [ ]:
oni_2026 = oni[oni["year"] == 2026]

In [ ]:
oni_2026

### Your turn 2
1. Print the unique city names in `rain`.
2. How many distinct years are in `oni`? Use `nunique()`.

In [ ]:
# Your code here

In [ ]:
# Your code here

## 3. Clean habits

These two files are already tidy, but we still run the missing-value and duplicate checks. In real work, skipping this step is how silent bugs sneak in.

In [ ]:
oni.isnull().sum()

In [ ]:
rain.isnull().sum()

In [ ]:
print("Duplicate ONI rows:", oni.duplicated().sum())

In [ ]:
print("Duplicate rain rows:", rain.duplicated().sum())

**What we learn:** no missing values and no full-row duplicates. We do not need to drop or fill anything today. Write that decision down in your notes: "checked; nothing to fix."

If rainfall ever had a missing month, we would decide whether to drop that month, leave it as missing, or (later courses) impute. For now, the habit matters more than a fancy fix.

## 4. Filter and transform

We want separate city views, then rainy-season months, then ENSO labels.

First, filter each city into its own table.

In [ ]:
lagos = rain[rain["city"] == "Lagos"]

In [ ]:
print("Lagos rows:", len(lagos))

In [ ]:
kano = rain[rain["city"] == "Kano"]

In [ ]:
print("Kano rows:", len(kano))

Nigeria's main rainy season for this mission is **JJAS**: June, July, August, September. Filter Kano to those months.

In [ ]:
kano_jjas = kano[kano["month"].isin([6, 7, 8, 9])]

In [ ]:
print("Kano JJAS rows:", len(kano_jjas))

In [ ]:
kano_jjas.head()

### Your turn 3
Filter Lagos to JJAS months (6, 7, 8, 9). Store the result as `lagos_jjas` and print how many rows it has.

In [ ]:
# Your code here

### Create ENSO labels with `.loc`

We start every row as Neutral, then overwrite El Nino and La Nina with boolean masks. No loops. No `apply`.

In [ ]:
oni_labeled = oni.copy()

In [ ]:
oni_labeled["enso"] = "Neutral"

In [ ]:
oni_labeled.loc[oni_labeled["oni"] >= 0.5, "enso"] = "El Nino"

In [ ]:
oni_labeled.loc[oni_labeled["oni"] <= -0.5, "enso"] = "La Nina"

In [ ]:
oni_labeled["enso"].value_counts()

Optional strength bands for El Nino seasons. We only label El Nino rows more finely; everything else stays `Other` for this column.

In [ ]:
oni_labeled["strength"] = "Other"

In [ ]:
oni_labeled.loc[oni_labeled["oni"] >= 0.5, "strength"] = "Moderate El Nino"

In [ ]:
oni_labeled.loc[oni_labeled["oni"] >= 1.5, "strength"] = "Strong El Nino"

In [ ]:
oni_labeled["strength"].value_counts()

### Your turn 4
Using `oni_labeled`, show only the rows where `enso` is `"El Nino"` and `year` is 2026. Which seasons already cross the 0.5 threshold?

In [ ]:
# Your code here

## 5. Merge ONI with rainfall

We join rainfall to ONI on the shared keys `year` and `month`.

**Talking point for yourself:** before you merge, ask:
1. What does one row mean in each table?
2. Which columns are the keys?
3. Should this be one-to-one, one-to-many, or many-to-one?
4. How many rows do I expect after the merge?

Here, each Lagos (or Kano) month should match at most one ONI season row.

In [ ]:
print("Lagos before merge:", lagos.shape)

In [ ]:
print("ONI labeled:", oni_labeled.shape)

Keep the useful ONI columns, then merge.

In [ ]:
oni_keys = oni_labeled[["year", "month", "season", "oni", "enso", "strength"]]

In [ ]:
lagos_merged = lagos.merge(oni_keys, on=["year", "month"], how="inner")

In [ ]:
print("Lagos after merge:", lagos_merged.shape)

In [ ]:
lagos_merged.head()

Do the same for Kano.

In [ ]:
kano_merged = kano.merge(oni_keys, on=["year", "month"], how="inner")

In [ ]:
print("Kano after merge:", kano_merged.shape)

In [ ]:
kano_merged.head()

**Check the grain after merging.** We expect 912 rows per city if every rainfall month found an ONI match (1950-2025). If the count drops a lot, a key is wrong or a year range does not overlap.

Also check that we did not accidentally duplicate months:

In [ ]:
print("Lagos duplicate year-month:", lagos_merged.duplicated(subset=["year", "month"]).sum())

In [ ]:
print("Kano duplicate year-month:", kano_merged.duplicated(subset=["year", "month"]).sum())

### Your turn 5
Create `rain_merged` by merging the full `rain` table (both cities) with `oni_keys` on `year` and `month` (inner join). Print the shape. How does it compare with 1824 (the original rain row count)?

In [ ]:
# Your code here

## 6. Groupby: answer the mission

Restrict to JJAS, then compare mean and median rainfall by ENSO phase.

In [ ]:
kano_jjas_m = kano_merged[kano_merged["month"].isin([6, 7, 8, 9])]

In [ ]:
print("Kano JJAS merged rows:", len(kano_jjas_m))

In [ ]:
kano_jjas_m.groupby("enso")["precip_mm"].agg(["mean", "median", "count"]).round(2)

Now Lagos, same season window.

In [ ]:
lagos_jjas_m = lagos_merged[lagos_merged["month"].isin([6, 7, 8, 9])]

In [ ]:
print("Lagos JJAS merged rows:", len(lagos_jjas_m))

In [ ]:
lagos_jjas_m.groupby("enso")["precip_mm"].agg(["mean", "median", "count"]).round(2)

**Read the tables slowly.**
- Look at the **count** column first: are the groups large enough to discuss?
- Compare **El Nino** mean/median with **La Nina** and **Neutral**.
- Ask whether the North (Kano) gap looks larger than the South (Lagos) gap.

Write one sentence in your notes before you plot anything.

### Your turn 6
For Kano only, repeat the groupby using **JAS** months (7, 8, 9) instead of JJAS. Do the El Nino versus La Nina means move in the same direction?

In [ ]:
# Your code here

## 7. Two light charts

Charts must answer a question. Today we only need two.

**Chart 1 question:** How has ONI moved into the current 2026 event?

In [ ]:
recent_oni = oni[oni["year"] >= 2015]

In [ ]:
plt.figure(figsize=(10, 4))

In [ ]:
plt.plot(recent_oni["year"] + (recent_oni["month"] - 1) / 12.0, recent_oni["oni"])

In [ ]:
plt.axhline(0.5, linestyle="--")

In [ ]:
plt.axhline(-0.5, linestyle="--")

In [ ]:
plt.title("ONI since 2015 (dashed lines at +/- 0.5)")

In [ ]:
plt.xlabel("Year")

In [ ]:
plt.ylabel("ONI (C)")

In [ ]:
plt.tight_layout()

In [ ]:
plt.show()

**Chart 2 question:** In Kano JJAS, how does mean rainfall compare across ENSO phases?

In [ ]:
kano_means = kano_jjas_m.groupby("enso")["precip_mm"].mean().reindex(["La Nina", "Neutral", "El Nino"])

In [ ]:
plt.figure(figsize=(6, 4))

In [ ]:
plt.bar(kano_means.index.astype(str), kano_means.values)

In [ ]:
plt.title("Kano JJAS mean rainfall by ENSO phase")

In [ ]:
plt.ylabel("Mean precip (mm)")

In [ ]:
plt.tight_layout()

In [ ]:
plt.show()

## 8. Findings and caveats

Use this template in your notes (and later in the homework paragraph):

1. **Finding:** In the historical record, Kano JJAS mean rainfall is lower in El Nino seasons than in La Nina seasons. Lagos shows a smaller gap.
2. **Support:** cite the groupby table (means, medians, counts).
3. **Caveat:** this is an association in monthly summaries. It is not proof that El Nino alone caused any single dry spell. Local weather, Atlantic patterns, and climate change also matter.
4. **Today:** ONI has climbed through 2026 into very strong values. That raises the chance of El Nino-like impacts, but it does not guarantee them for every Nigerian city.

FEWS NET and local reporting often highlight northern dry-spell risk under El Nino. Our Kano numbers lean the same way historically. Still speak carefully.

## 9. Challenge (10 to 15 minutes)

Choose **one** option.

### Option A (recommended)
Using `kano_merged`, keep JAS months (7, 8, 9) and El Nino rows only. Compare mean rainfall for `Moderate El Nino` versus `Strong El Nino` using the `strength` column.

### Option B (stretch)
Load `nigeria_north_staple_prices_monthly.csv`. Filter to `commodity == "Millet"`. Merge with `oni_keys` on `year` and `month`. Group the median USD price (`median_price_per_kg_usd`) by `enso`. Then write two sentences on **why this result is harder to trust** than the rainfall result (think inflation, exchange rates, market coverage, and short history).

In [ ]:
# Challenge: your code here (Option A or B)

## Recap
- An analytical workflow starts with a **question**, not with a random chart.
- `read_csv` loads tables; `shape`, `info`, `isnull`, and `duplicated` are the inspect habit.
- Boolean masks and `.loc` create labels without loops.
- `merge` needs clear keys and a grain check (row counts before and after).
- `groupby` compares groups that answer the question.
- Findings need **caveats**: association is not causation.
- Next session (W4S2) goes deeper on choosing charts that each answer a stated question.

## Homework
**Due:** before the next class. **Submit:** this notebook with Your turns completed, plus a short findings paragraph.

1. Finish **Your turns 1 to 6**.
2. Complete **Challenge Option A or B**.
3. Write a short findings paragraph (about 8 to 12 lines) that answers the mission question for Kano and Lagos, includes at least one number from your tables, and ends with one clear caveat.
4. Skim the EDA project brief below and note which dataset idea you might use (El Nino data is one option, not required).

## EDA PROJECT kickoff (starts this week)

This is the major Week 4 project. Structure:

**Problem -> Understanding -> Cleaning -> Exploration -> Viz -> Findings -> Conclusion**

**Deliverables**
- A notebook that follows the structure above
- A cleaned dataset (CSV)
- A short README
- A short findings report (can be a markdown section)

**Dataset choice is open.** Pick a question you care about and a public CSV you can load locally. The El Nino files from today are **one option**, not a requirement. Other ideas: a sport dataset, local market prices, education scores, a Kaggle beginner table, or a government open-data CSV.

**Rules of the road**
- Start with a clear question.
- Inspect before you clean.
- Every chart must answer a question.
- State caveats. Do not overclaim.

We will keep building this through Week 4 Session 2 (visualization and EDA). Come to the next class with a draft question and a candidate dataset name.